# RoadVision AI — Model Training and Tuning

Experiment with model variants, image sizes, batch sizes and hyperparameters, starting from the baseline in `06_yolo_baseline_training.ipynb`.

**Protocol (constraints):**
- One factor changes per experiment relative to the baseline (YOLOv8n, imgsz 640, batch 16, 30 epochs, patience 10, seed 42). Same split, same seed, same default augmentation.
- Selection metric: validation mAP@50–95 (also macro over classes with at least 20 val instances).
- No test split exists. Selecting on validation and reporting on validation is optimistic; do not present these numbers as final performance.
- Each finished run writes `done.json` and is skipped on rerun. Set `RV_FORCE=1` to retrain.

**Env vars:** `RV_GROUPS` (default `baseline,model,imgsz,batch,hyp`), `RV_FORCE`, `RV_RUN_FINAL`, `RV_TUNE`.

In [1]:
# %pip install -q ultralytics

In [2]:
from pathlib import Path
import os, json, time, random, platform, gc, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, ultralytics
from ultralytics import YOLO
from IPython.display import display

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Python:", platform.python_version(), "| PyTorch:", torch.__version__, "| Ultralytics:", ultralytics.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

DEFAULT_ROOT = Path.home() / ".cache/kagglehub/datasets/ashfakyeafi/road-vehicle-images-dataset/versions/2/roadvision_cleaned"
CLEAN_ROOT = Path(os.getenv("ROADVISION_CLEANED_ROOT", str(DEFAULT_ROOT))).expanduser().resolve()
if not CLEAN_ROOT.is_dir():
    raise FileNotFoundError(f"Prepared dataset not found: {CLEAN_ROOT}. Set ROADVISION_CLEANED_ROOT.")
DATA_YAML = CLEAN_ROOT / "roadvision_prepared.yaml"  # produced by 04_dataset_preparation.ipynb
if not DATA_YAML.is_file():
    raise FileNotFoundError(f"{DATA_YAML} not found. Run 04_dataset_preparation.ipynb first.")
_cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
_names = _cfg["names"]
CLASS_NAMES = [_names[k] for k in sorted(_names, key=int)] if isinstance(_names, dict) else list(_names)
NC = len(CLASS_NAMES)
SPLITS = {s: CLEAN_ROOT / s for s in ("train", "valid", "test")}
RUNS_DIR = CLEAN_ROOT / "training_runs"
REPORT_DIR = CLEAN_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
WORKERS = 2
MIN_EVAL_INSTANCES = 20  # classes with fewer val instances give statistically meaningless AP

def count_instances(label_dir, nc):
    counts = np.zeros(nc, dtype=int)
    for f in Path(label_dir).glob("*.txt"):
        for line in f.read_text(encoding="utf-8").splitlines():
            parts = line.split()
            if len(parts) == 5: counts[int(float(parts[0]))] += 1
    return counts

VAL_COUNTS = count_instances(SPLITS["valid"] / "labels", NC)
print("Dataset:", CLEAN_ROOT, "| classes:", NC)
print("Classes absent from val:", [CLASS_NAMES[i] for i in range(NC) if VAL_COUNTS[i] == 0])

Python: 3.10.22 | PyTorch: 2.14.1+cu130 | Ultralytics: 8.4.193
CUDA: True NVIDIA GeForce RTX 3060 Laptop GPU
Dataset: /root/.cache/kagglehub/datasets/ashfakyeafi/road-vehicle-images-dataset/versions/2/roadvision_cleaned | classes: 21
Classes absent from val: ['ambulance', 'army vehicle', 'auto rickshaw', 'garbagevan', 'human hauler']


## 1. Experiment definitions
Keys `model`, `imgsz`, `batch`, `epochs`, `patience` override `BASE`; any other key is passed unchanged to `model.train()`.

In [3]:
BASE = dict(model="yolov8n.pt", imgsz=640, batch=16, epochs=30, patience=10)

GROUPS = {
    "baseline": {"baseline": {}},
    "model": {
        "yolov8s": {"model": "yolov8s.pt"},
        "yolov9c": {"model": "yolov9c.pt"},
    },
    "imgsz": {
        "img800": {"imgsz": 800},
        "img960": {"imgsz": 960},
    },
    "batch": {
        "batch8": {"batch": 8},
        "batch32": {"batch": 32},
    },
    "hyp": {
        "lr0_0.005": {"lr0": 0.005},
        "lr0_0.02": {"lr0": 0.02},
        "adamw_1e-3": {"optimizer": "AdamW", "lr0": 0.001},
        "cos_lr": {"cos_lr": True},
        "cls_1.0": {"cls": 1.0},
        "scale_0.7": {"scale": 0.7},
    },
}
SELECTED = [g.strip() for g in os.getenv("RV_GROUPS", "baseline,model,imgsz,batch,hyp").split(",") if g.strip()]
FORCE = os.getenv("RV_FORCE", "0") == "1"
unknown = [g for g in SELECTED if g not in GROUPS]
assert not unknown, f"Unknown groups: {unknown}"
for g in SELECTED:
    for n, o in GROUPS[g].items(): print(f"{g:9s} {n:12s} {o}")

baseline  baseline     {}
model     yolov8s      {'model': 'yolov8s.pt'}
model     yolov9c      {'model': 'yolov9c.pt'}
imgsz     img800       {'imgsz': 800}
imgsz     img960       {'imgsz': 960}
batch     batch8       {'batch': 8}
batch     batch32      {'batch': 32}
hyp       lr0_0.005    {'lr0': 0.005}
hyp       lr0_0.02     {'lr0': 0.02}
hyp       adamw_1e-3   {'optimizer': 'AdamW', 'lr0': 0.001}
hyp       cos_lr       {'cos_lr': True}
hyp       cls_1.0      {'cls': 1.0}
hyp       scale_0.7    {'scale': 0.7}


## 2. Run function
Trains, then re-evaluates `best.pt` on validation. An out-of-memory error is recorded and the queue continues.

In [4]:
def summarize(m):
    box = m.box
    ap = {int(c): box.class_result(k) for k, c in enumerate(box.ap_class_index)}  # (P, R, AP50, AP50-95)
    ev = [c for c in range(NC) if VAL_COUNTS[c] >= MIN_EVAL_INSTANCES and c in ap]
    return {
        "P": float(box.mp), "R": float(box.mr), "mAP50": float(box.map50), "mAP50-95": float(box.map),
        "macro_mAP50_eval": float(np.mean([ap[c][2] for c in ev])) if ev else float("nan"),
        "macro_mAP50-95_eval": float(np.mean([ap[c][3] for c in ev])) if ev else float("nan"),
        "n_eval_classes": len(ev), "ms_inference": float(m.speed["inference"]),
    }

def read_history(run_dir):
    p = Path(run_dir) / "results.csv"
    if not p.is_file(): return None
    h = pd.read_csv(p); h.columns = h.columns.str.strip(); return h

def run_experiment(group, name, overrides, plots=False):
    run = "yolov8n_baseline" if name == "baseline" else f"exp_{name}"
    run_dir = RUNS_DIR / run
    done = run_dir / "done.json"
    if done.is_file() and not FORCE:
        print(f"[skip] {run} (done)"); return json.loads(done.read_text())
    cfg = {**BASE, **overrides}
    extra = {k: v for k, v in cfg.items() if k not in ("model", "imgsz", "batch", "epochs", "patience")}
    row = {"group": group, "name": name, "run": run, "model": cfg["model"], "imgsz": cfg["imgsz"], "batch": cfg["batch"],
           "epochs_cfg": cfg["epochs"], "overrides": json.dumps(overrides), "status": "ok"}
    try:
        best = run_dir / "weights" / "best.pt"
        if name == "baseline" and best.is_file() and not FORCE:
            print("[reuse] baseline from 06 (no retraining)")
            train_min = None
        else:
            print(f"[train] {run}: {cfg}")
            t0 = time.time()
            model = YOLO(cfg["model"])
            model.train(data=str(DATA_YAML), imgsz=cfg["imgsz"], epochs=cfg["epochs"], batch=cfg["batch"], patience=cfg["patience"],
                        workers=WORKERS, device=DEVICE, seed=SEED, deterministic=True, pretrained=True,
                        project=str(RUNS_DIR), name=run, exist_ok=True, plots=plots, verbose=False, **extra)
            run_dir = Path(model.trainer.save_dir); best = run_dir / "weights" / "best.pt"
            train_min = (time.time() - t0) / 60
        h = read_history(run_dir)
        if train_min is None and h is not None and "time" in h: train_min = float(h["time"].iloc[-1]) / 60
        best_model = YOLO(str(best))
        m = best_model.val(data=str(DATA_YAML), split="val", imgsz=cfg["imgsz"], batch=cfg["batch"], device=DEVICE,
                           workers=WORKERS, plots=False, verbose=False, project=str(RUNS_DIR), name=f"{run}_val", exist_ok=True)
        row.update(summarize(m))
        row.update({"epochs_run": None if h is None else len(h), "train_min": train_min,
                    "params_M": sum(p.numel() for p in best_model.model.parameters()) / 1e6, "best_weights": str(best)})
        done.write_text(json.dumps(row, indent=2), encoding="utf-8")
    except torch.cuda.OutOfMemoryError:
        row["status"] = "OOM"; print(f"[OOM] {run}: reduce batch or imgsz")
    except Exception as e:
        row["status"] = f"error: {type(e).__name__}: {e}"; print(f"[error] {run}: {e}")
    finally:
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return row

## 3. Run the selected experiments
Each run is at least as long as the baseline (632 s for YOLOv8n at 640 on an RTX 3060). `yolov9c` and `img960` take longer. Restrict with `RV_GROUPS`.

In [ ]:
for g in SELECTED:
    for n, o in GROUPS[g].items():
        r = run_experiment(g, n, o)
        print({k: r.get(k) for k in ("name", "status", "mAP50-95", "macro_mAP50-95_eval", "train_min")})

[reuse] baseline from 06 (no retraining)
Ultralytics 8.4.193 🚀 Python-3.10.22 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 11913MiB)
Model summary (fused): 73 layers, 3,137,567 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1050.9±279.6 MB/s, size: 37.6 KB)
val: Scanning /root/.cache/kagglehub/datasets/ashfakyeafi/road-vehicle-images-dataset/versions/2/roadvision_cleaned/valid/labels.cache... 300 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 300/300 57.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 8.0it/s 2.4s0.1ss
                   all        300       2560      0.691      0.383      0.436      0.269
Speed: 0.4ms preprocess, 1.9ms inference, 0.0ms loss, 1.6ms postprocess per image
{'name': 'baseline', 'status': 'ok', 'mAP50-95': 0.26853803527930165, 'macro_mAP50-95_eval': 0.3055366760378156, 'train_min': 11.177249999999999}
[train] exp_yolov8s

## 4. Compare experiments
Delta columns are relative to the baseline row.

In [ ]:
def collect_results():
    rows = [json.loads(p.read_text()) for p in RUNS_DIR.glob("*/done.json")]
    return pd.DataFrame(rows)

res = collect_results()
if res.empty:
    print("No finished experiments.")
else:
    base = res[res["name"] == "baseline"]
    if not base.empty:
        for col in ("mAP50-95", "macro_mAP50-95_eval", "R"):
            res[f"d_{col}"] = res[col] - float(base[col].iloc[0])
    res = res.sort_values("mAP50-95", ascending=False).reset_index(drop=True)
    cols = ["name", "group", "model", "imgsz", "batch", "overrides", "epochs_run", "train_min", "params_M",
            "P", "R", "mAP50", "mAP50-95", "macro_mAP50-95_eval", "d_mAP50-95", "ms_inference", "status"]
    display(res[[c for c in cols if c in res.columns]].round(4))
    res.to_csv(REPORT_DIR / "experiment_comparison.csv", index=False)
    ax = res.set_index("name")["mAP50-95"].sort_values().plot.barh(figsize=(8, max(3, 0.4 * len(res))))
    if not base.empty: ax.axvline(float(base["mAP50-95"].iloc[0]), color="k", ls="--", lw=1, label="baseline")
    ax.set_xlabel("val mAP50-95"); ax.legend(); plt.tight_layout(); plt.show()

## 5. Optional: Ultralytics hyperparameter evolution
Genetic search over training hyperparameters (`RV_TUNE=1`). Slow: it trains `RV_TUNE_ITERS` times. Output: `best_hyperparameters.yaml` in the tune directory.

In [ ]:
if os.getenv("RV_TUNE", "0") == "1":
    tuner = YOLO(BASE["model"])
    tuner.tune(data=str(DATA_YAML), imgsz=BASE["imgsz"], batch=BASE["batch"], device=DEVICE, workers=WORKERS, seed=SEED,
               epochs=int(os.getenv("RV_TUNE_EPOCHS", 30)), iterations=int(os.getenv("RV_TUNE_ITERS", 20)),
               optimizer="AdamW", plots=False, save=False, val=False, project=str(RUNS_DIR), name="tune", exist_ok=True)
    print("Results:", RUNS_DIR / "tune")
else:
    print("Skipped (set RV_TUNE=1).")

## 6. Final run
Combine only settings that improved over the baseline in section 4, and edit `FINAL_OVERRIDES`. Combinations are not validated by the one-factor experiments; the final run is their test. Enable with `RV_RUN_FINAL=1`.

In [ ]:
RUN_FINAL = os.getenv("RV_RUN_FINAL", "0") == "1"
FINAL_OVERRIDES = {"epochs": 100, "patience": 20}  # add winners, e.g. {"model": "yolov8s.pt", "imgsz": 800, "cls": 1.0}
if RUN_FINAL:
    final_row = run_experiment("final", "final", FINAL_OVERRIDES, plots=True)
    print({k: final_row.get(k) for k in ("status", "P", "R", "mAP50", "mAP50-95", "macro_mAP50-95_eval", "best_weights")})
else:
    print("Skipped (set RV_RUN_FINAL=1).")

## 7. Notes
- `08_model_evaluation.ipynb` loads the best weights from `reports/experiment_comparison.csv` (override with `RV_WEIGHTS_PATH`).
- Differences below about 0.01 mAP50-95 are within run-to-run noise for a single seed; confirm winners with a second seed before concluding.
- Report the test split only once it exists and only after the configuration is final.